# WDL — Module 1 · Session 4 — Reading a river gauge record
### Reading notebook

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 1 — The Water Cycle as Data |
| **Session** | Session 4 of 30 |
| **CO** | CO1 |
| **Topics** | River stage; warning and danger levels; rainfall–river response; lag time; rise and recession |
| **Water hook** | *When a river crosses the danger mark* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
Rain gauges tell us what falls from the sky. Floods happen in rivers. At a river **gauge station**, a tall marked scale
(or an automatic sensor) records the **water level**, called the **stage**, in metres. Each station has two key lines:
the **warning level** (get ready) and the **danger level** (water may spill over the banks).
In India, CWC and state water resources departments run these stations.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from IPython.display import display, Markdown
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C"
def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def close(a, b, tol=0.02):  # within 2% (or 0.01 for very small numbers)
    try: return abs(float(a) - float(b)) <= max(abs(float(b)) * tol, 0.01)
    except (TypeError, ValueError): return False

DATA_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/M1/sample_station_2023.csv"
def _make_sample_station():
    rng = np.random.default_rng(2023)
    days = pd.date_range("2023-01-01", "2023-12-31", freq="D")
    p_wet  = [0.03, 0.03, 0.08, 0.20, 0.35, 0.40, 0.45, 0.48, 0.55, 0.50, 0.30, 0.08]
    mean_r = [3, 3, 6, 9, 10, 7, 7, 8, 11, 11, 9, 4]
    rain = []
    for d in days:
        m = d.month - 1
        rain.append(rng.gamma(0.8, mean_r[m] / 0.8) if rng.random() < p_wet[m] else 0.0)
    rain = np.round(np.array(rain), 1)
    rain[days.get_loc(pd.Timestamp("2023-09-21"))] = 96.4
    rain[days.get_loc(pd.Timestamp("2023-09-22"))] = 58.2
    level, soil, channel = [], 0.0, 0.0
    for r in rain:
        soil = soil * 0.6 + max(r - 5, 0) * 0.5
        channel = channel * 0.82 + soil * 0.022
        level.append(round(1.8 + channel, 2))
    return pd.DataFrame({"date": days, "rain_mm": rain, "level_m": level})
try:
    station = pd.read_csv(DATA_URL, parse_dates=["date"])
except Exception:
    station = _make_sample_station()   # works offline too: same numbers
station = station.set_index("date")
rainfall, river = station[["rain_mm"]], station[["level_m"]]
WARNING_LEVEL, DANGER_LEVEL = 3.6, 4.2
display(Markdown("✅ **Ready.** Sample station loaded: one year (2023) of daily rainfall and river level. "
                 "*Illustrative, synthetic data for practice; not an official IMD or CWC record.*"))

---
## Part 1 · Rain and river, side by side
Press ▶ to plot the river level under the same year's rainfall.

In [ ]:
#@title ▶ Plot rainfall and river level together
fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(11, 6.5), gridspec_kw={"height_ratios": [1, 1.6]})
a1.bar(rainfall.index, rainfall.rain_mm, width=1.0, color=BLUE); a1.set_ylabel("Rain (mm/day)")
a1.set_title("Rain falls first; the river responds after")
lv = river.level_m
a2.plot(lv.index, lv, color="black", lw=1.4, label="River level")
a2.axhline(WARNING_LEVEL, color=ORANGE, ls="--", lw=1.3, label=f"Warning level ({WARNING_LEVEL} m)")
a2.axhline(DANGER_LEVEL, color=RED, ls="--", lw=1.3, label=f"Danger level ({DANGER_LEVEL} m)")
a2.fill_between(lv.index, DANGER_LEVEL, lv, where=lv > DANGER_LEVEL, color=RED, alpha=0.25)
a2.text(lv.index[5], WARNING_LEVEL + 0.04, f"Warning level {WARNING_LEVEL} m", color=ORANGE, fontsize=10)
a2.text(lv.index[5], DANGER_LEVEL + 0.04, f"Danger level {DANGER_LEVEL} m", color=RED, fontsize=10)
a2.set_ylabel("Water level (m)")
a2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.tight_layout(); plt.show()

### What to notice
- **January to March:** no rain, and the river sits flat. This is **base flow**: groundwater slowly seeping into the river.
- **Small rains barely move the river.** Dry soil soaks up light rain first.
- **The big September storm** pushes the river above the danger level.

---
## Part 2 · Lag time
Zoom in on the September storm. Press ▶.

In [ ]:
#@title ▶ Zoom in on the flood { run: "auto" }
days_either_side = 12 #@param {type:"slider", min:5, max:30, step:1}
peak_rain_day = rainfall.rain_mm.idxmax()
win = slice(peak_rain_day - pd.Timedelta(days=days_either_side), peak_rain_day + pd.Timedelta(days=days_either_side))
r, lv = rainfall.rain_mm[win], river.level_m[win]
peak_level_day = lv.idxmax()
fig, ax = plt.subplots(figsize=(11, 4.5)); ax2 = ax.twinx()
ax.bar(r.index, r, color=BLUE, alpha=0.6, width=0.8); ax.set_ylabel("Rain (mm/day)", color=BLUE)
ax2.plot(lv.index, lv, "k-o", ms=3); ax2.set_ylabel("River level (m)")
ax2.axhline(DANGER_LEVEL, color=RED, ls="--", lw=1.2)
ax2.text(lv.index[0], DANGER_LEVEL + 0.04, "Danger level", color=RED, fontsize=10)
ax2.annotate("Peak level", xy=(peak_level_day, lv.max()), xytext=(15, -5), textcoords="offset points")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
lag = (peak_level_day - peak_rain_day).days
ax.set_title(f"Heaviest rain {peak_rain_day:%d %b}, river peak {peak_level_day:%d %b}: lag time {lag} days")
plt.tight_layout(); plt.show()

**Lag time** is the delay between the heaviest rain and the highest river level. Water has to run off the land, gather in
streams and travel down to the gauge. Lag time is precious: it is the window in which flood warnings can be issued.

Notice too that the river **rises fast and falls slowly**. The falling part is called the **recession**.

---
## Part 3 · How long was the river in danger?
Move the danger level and see how the count of dangerous days changes. Press ▶.

In [ ]:
#@title ▶ Days above a level { run: "auto" }
level_to_test = 4.2 #@param {type:"slider", min:2.0, max:4.6, step:0.1}
lv = river.level_m
above = lv > level_to_test
display(Markdown(f"Days above **{level_to_test:.1f} m**: **{above.sum()}**. "
                 f"Highest level of the year: **{lv.max():.2f} m** on **{lv.idxmax():%d %B}**."))

### Expected output
At the official danger level (**4.2 m**) you should see **4 days**; at the warning level (**3.6 m**), **6 days**.

### 🤔 Think about it
1. Why is a danger level set by the local bank height rather than a single number for every river?
2. If the city upstream paved over its fields, would the lag time get shorter or longer? Why?

### ⏭️ Next
Open the **Practice activity** and measure the flood yourself.